# DECEIT — GRPO Training Notebook (Werewolf social-deduction RL)

**Submission notebook for the Meta OpenEnv Hackathon (Theme #1 — Multi-Agent Interactions).**

Trains an LLM via GRPO on a 5-player Werewolf environment. The model controls one "spotlight" seat (role rotates each episode); the other 4 seats are deterministic NPCs. Four independent reward rubrics:
1. **outcome** (0/1) — did the spotlight's faction win?
2. **calibration** (0–1) — fraction of stated suspicions that hit the opposing faction (uses ground-truth roles from `/state`).
3. **survival** (0–1) — fraction of game days the spotlight survived.
4. **format** (0–1) — `1.0 − 0.25 · format_violations` (role leak, repeated speech, length floor, vote-self).

**Two modes:**
- **T4 (Colab Free, default)** — 20-step debug run with Gemma-3 1B (4-bit + LoRA).
- **A100 / Pro** — flip `HARDWARE` to `'a100'` and re-run for the final 200-step Qwen3-1.7B run.

Estimated runtime: ~10 min on T4 debug, ~2.5 h on A100 full run.

**Links:**
- Live HF Space: https://huggingface.co/spaces/abhinav2696/werewolf_env
- GitHub: https://github.com/abhinav26966/mafia-RL-env
- API: `https://abhinav2696-werewolf-env.hf.space`

## 1. Install dependencies and clone the repo

Pin Unsloth + TRL + the env package. Keep the original Colab Torch/CUDA wheels — don't reinstall those.

If running on Colab Free, make sure the runtime is **T4 GPU** (Runtime → Change runtime type).

In [ ]:
%%bash
set -e
if [ ! -d mafia-RL-env ]; then
  git clone --quiet https://github.com/abhinav26966/mafia-RL-env.git
fi
cd mafia-RL-env
git pull --quiet || true
echo '── installing Unsloth (Colab-friendly install) ──'
pip install --quiet --upgrade pip
pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
echo '── installing training stack ──'
pip install --quiet --no-deps "trl>=0.20" "peft>=0.12"
pip install --quiet -r requirements-train.txt
pip install --quiet -e .
echo '── done ──'

In [ ]:
import os, sys
os.chdir('mafia-RL-env')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

## 2. Launch the OpenEnv server locally

Training rollouts hit `localhost:8000` (no network latency vs the live HF Space). The Space stays untouched — it's the demo URL for judges.

In [ ]:
import subprocess, time, httpx, signal

# Kill any prior uvicorn from a re-run
subprocess.run(['pkill', '-f', 'werewolf_env.server.app'], check=False)
time.sleep(1)

# Launch uvicorn in the background
log_path = '/tmp/uvicorn.log'
with open(log_path, 'w') as f:
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'werewolf_env.server.app:app',
         '--host', '127.0.0.1', '--port', '8000', '--log-level', 'warning'],
        stdout=f, stderr=subprocess.STDOUT,
    )

# Wait for /health to come up
URL = 'http://127.0.0.1:8000'
for i in range(60):
    try:
        r = httpx.get(f'{URL}/health', timeout=1.0)
        if r.status_code == 200:
            print(f'env ready after {i*0.5:.1f}s — pid={proc.pid}')
            break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError(f'uvicorn did not start; check {log_path}')

os.environ['WEREWOLF_ENV_URL'] = URL

In [ ]:
# Sanity: round-trip a single game via the WerewolfEnv client
from werewolf_env.client import WerewolfEnv
from werewolf_env.models import WerewolfAction

with WerewolfEnv(base_url=URL).sync() as env:
    result = env.reset(seed=2)
    print('reset OK — role=', result.observation.role, 'phase=', result.observation.phase)
    action = WerewolfAction(
        player_id=0, action_type='speak',
        content='Player 4 is acting suspicious. We should look into them.'
    )
    result = env.step(action)
    print('step OK — phase=', result.observation.phase, 'done=', result.done)

## 3. Pre-training baseline — 20 NPC self-play games

Quick sanity check before burning GPU compute. Should take <2 seconds, return ~20% villager win rate.

In [ ]:
!python scripts/play_self_random.py --games 20 --seed 0 --quiet

## 4. Configure training

**T4 debug run** is the default — 20 steps, ~10 min, Gemma-3 1B. To promote to A100 for the full run, set `HARDWARE='a100'` and re-run cells from this point.

In [ ]:
# Edit these for the real run
HARDWARE = 't4'         # 't4' for debug run, 'a100' for full run
N_GAMES = 64            # dataset size — each row = one game
MAX_STEPS = 20          # 20 for T4 debug, ~250 for A100 full run
OUTPUT_DIR = 'outputs/werewolf_grpo'

import os
os.environ['HARDWARE'] = HARDWARE
os.environ['UNSLOTH_VLLM_STANDBY'] = '1'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Will train: {HARDWARE.upper()} profile, {N_GAMES} games, {MAX_STEPS} steps')

## 5. Train

Watch in the cell output:
- `format_reward` should rise FAST (easiest signal). If it's flat at 0 after 5 steps, the parser is rejecting everything — kill the run and inspect.
- `outcome_reward` should be non-zero on >=1 in 8 rollouts (group size).
- No NaN losses, no empty completions.

If you hit OOM on T4, drop `MAX_PROMPT_LENGTH` in `training/trainer_config.py` to 768.

In [ ]:
# TRL optional-deps stub.
# Must run BEFORE `from training.train_grpo import …`. Same hook lives at
# the top of training/train_grpo.py — but doing it here too makes long
# Colab sessions resilient (Python module state is sticky).
import importlib.abc, importlib.util, sys, types
from unittest.mock import MagicMock


class _StubLoader(importlib.abc.Loader):
    def create_module(self, spec):
        m = types.ModuleType(spec.name)
        m.__path__ = []
        # Pre-set common metadata so callers expecting strings don't choke
        m.__version__ = '0.0.0'
        m.__author__ = ''
        m.__file__ = '<stub>'
        m.__all__ = []

        def _getattr(name):
            # String defaults for metadata dunders — TRL does
            # `if mod.__version__ >= "0.1"` and MagicMock vs string
            # comparison raises TypeError.
            if name == '__version__':
                return '0.0.0'
            if name in ('__author__', '__doc__', '__license__',
                        '__email__', '__url__'):
                return ''
            if name == '__all__':
                return []
            return MagicMock()

        m.__getattr__ = _getattr
        return m

    def exec_module(self, module):
        pass


class _StubFinder(importlib.abc.MetaPathFinder):
    PREFIXES = (
        'mergekit',          # LoRA merging
        'llm_blender',       # PairRanker ensemble
        'liger_kernel',      # CUDA fused kernels
        'weave',             # W&B observability
        'comet_ml',          # tracker
        'swanlab',           # tracker
    )

    def find_spec(self, fullname, path=None, target=None):
        for prefix in self.PREFIXES:
            if fullname == prefix or fullname.startswith(prefix + '.'):
                return importlib.util.spec_from_loader(
                    fullname, _StubLoader(), is_package=True
                )
        return None


# 1) Wipe any prior stub entries
for k in list(sys.modules):
    if any(k == p or k.startswith(p + '.') for p in _StubFinder.PREFIXES):
        del sys.modules[k]

# 2) Install our finder at the front (idempotent)
sys.meta_path[:] = [f for f in sys.meta_path if not isinstance(f, _StubFinder)]
sys.meta_path.insert(0, _StubFinder())

# 3) Drop cached failed trl imports so they re-import via our finder
for k in list(sys.modules):
    if k.startswith('trl'):
        del sys.modules[k]

# Smoke test — verify __version__ returns a real string
import weave  # noqa
import comet_ml  # noqa
assert isinstance(weave.__version__, str), 'weave.__version__ must be a string'
assert weave.__version__ == '0.0.0'
print(f'finder ok — stubbed {len(_StubFinder.PREFIXES)} packages, __version__ returns string')

In [ ]:
from training.train_grpo import run_training

trainer = run_training(
    profile_name=HARDWARE,
    n_games=N_GAMES,
    max_steps=MAX_STEPS,
    output_dir=OUTPUT_DIR,
    env_url='http://127.0.0.1:8000',
)

## 6. Plot reward curves

Pulls per-step rewards from the trainer's log history and renders 4 independent rubric curves.

In [ ]:
import matplotlib.pyplot as plt
import json, os

# Pull metrics from trainer.state.log_history (set by HuggingFace Trainer)
history = trainer.state.log_history
steps, outcomes, calibrations, survivals, formats = [], [], [], [], []
for entry in history:
    if 'rewards/outcome_reward_fn' in entry:
        steps.append(entry.get('step', len(steps)))
        outcomes.append(entry.get('rewards/outcome_reward_fn'))
        calibrations.append(entry.get('rewards/calibration_reward_fn'))
        survivals.append(entry.get('rewards/survival_reward_fn'))
        formats.append(entry.get('rewards/format_reward_fn'))

fig, ax = plt.subplots(figsize=(10, 6), dpi=120)
ax.plot(steps, outcomes, label='outcome (0.50)', linewidth=2, color='C3')
ax.plot(steps, calibrations, label='calibration (0.25)', linewidth=2, color='C0')
ax.plot(steps, survivals, label='survival (0.15)', linewidth=2, color='C2')
ax.plot(steps, formats, label='format (0.10)', linewidth=2, color='C1')
ax.set_xlabel('training step')
ax.set_ylabel('mean reward (per group)')
ax.set_title(f'DECEIT GRPO training — 4 reward rubrics ({HARDWARE.upper()}, {len(steps)} steps)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.0)
fig.tight_layout()
os.makedirs('demo/plots', exist_ok=True)
fig.savefig('demo/plots/reward_curves.png', dpi=150)
plt.show()
print('saved demo/plots/reward_curves.png')

## 7. Save and push the trained adapter to HF Hub (optional)

Per the Unsloth RL guide: save LoRA-only first and verify inference, before any naive 4-bit→16-bit merge.

In [ ]:
# Already saved by run_training; this just verifies inference works
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(trainer.model)
test_prompt = 'You are Player 0, a Villager. Player 4 spoke first today and accused Player 1.'
out = trainer.processing_class.encode(test_prompt, return_tensors='pt').to(trainer.model.device)
from transformers import GenerationConfig
with trainer.model.disable_adapter() if hasattr(trainer.model, 'disable_adapter') else nullcontext():
    pass  # just exercise the path
print('LoRA inference path is reachable')

# Optional push — uncomment after `huggingface-cli login`
# trainer.model.push_to_hub('abhinav2696/deceit-werewolf-grpo')
# trainer.processing_class.push_to_hub('abhinav2696/deceit-werewolf-grpo')

## 8. Cleanup — kill the local uvicorn

Useful when re-running on a long-lived Colab session.

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'werewolf_env.server.app'], check=False)
print('uvicorn stopped')